# 08. AI Watermarking, AI Detection, and Model Obfuscation

- **Category:** 11_ai_safety_privacy_and_verification
- **Lineage & Citations:** Kirchenbauer et al. (2023 *A Watermark for Large Language Models*); Mitchell et al. (2023 *DetectGPT: Zero-Shot Machine-Generated Text Detection using Probability Curvature*); Bao et al. (2023 *Fast-DetectGPT: Efficient Zero-Shot Detection of Machine-Generated Text via Conditional Probability Curvature*); Fan et al. (2019 *Rethinking Deep Neural Network Ownership Verification: Embedding Passports to Defeat Ambiguity Attacks*).
- **Purpose:** Formulate and evaluate generative watermarking, synthetic content detection, and model intellectual property (IP) protection: Kirchenbauer green/red list logit biasing and one-tailed $z$-score hypothesis testing, Fast-DetectGPT 1-pass conditional likelihood curvature, and cryptographic Passport Layer key gating.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [07. Privacy DP SGD PATE Gradient Inversion DLG and MIA](07_privacy_dp_sgd_pate_gradient_inversion_dlg_and_mia.ipynb) $\to$ `🔗 Downstream Syntheses:` [09. OOD Detection Abstention Conformal and Algorithmic Fairness](09_ood_detection_abstention_conformal_and_algorithmic_fairness.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not apply static logit bias $\delta$ to low-entropy tokens (code syntax, math proofs, chemical SMILES) as it causes grammatical collapse; do not rely on naive uncalibrated perplexity thresholds for AI detection across non-native human text; do not place passport layers solely on final output projections.

## Mathematical Foundations & Governing Equations

1. **Green-List Logit Biasing**: $l' = l + \delta \cdot m_{\text{green}}, \quad m_{\text{green}} \in \{0, 1\}^V$
2. **One-Tailed $z$-Score**: $z = \frac{|S_G| - \gamma T}{\sqrt{T \gamma (1 - \gamma) + 10^{-8}}} > 4.0 \implies p < 3.2 \times 10^{-5}$
3. **Fast-DetectGPT Curvature**: $\mathbf{d} = \frac{1}{T} \sum_{t=1}^T (\log p_\theta(x_t \mid x_{<t}) - \log p_{\text{ref}}(x_t \mid x_{<t}))$
4. **Passport Layer Security Gate**: $s = \sigma(K \cdot W_{\text{pass}}), \quad y = (x W + b) \odot s$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/11_ai_safety_privacy_and_verification/08_ai_watermarking_ai_detection_and_model_obfuscation.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def watermark_logit_bias(logits, green_mask, delta):
    return logits + float(delta) * green_mask.unsqueeze(0)

def watermark_z_score(green_counts, seq_lens, gamma):
    expected = float(gamma) * seq_lens
    std = torch.sqrt(seq_lens * float(gamma) * (1.0 - float(gamma)) + 1e-8)
    return (green_counts - expected) / std

def fast_detect_gpt_score(log_probs, ref_log_probs):
    return torch.mean(log_probs - ref_log_probs, dim=-1)

def passport_layer_forward(x, weight, bias, passport_key, passport_weight):
    scale = torch.sigmoid(passport_key @ passport_weight)
    base = x @ weight + bias
    return base * scale.unsqueeze(0)

In [ ]:
# Parity verification against golden reference tensors
w_logits = watermark_logit_bias(weights["logits"], weights["green_mask"], delta=2.5)
assert torch.max(torch.abs(w_logits - weights["watermarked_logits_ref"])) < 1e-5

z_scores = watermark_z_score(weights["green_counts"], weights["seq_lens"], gamma=0.5)
assert torch.max(torch.abs(z_scores - weights["z_scores_ref"])) < 1e-5

detect_scores = fast_detect_gpt_score(weights["log_probs"], weights["ref_log_probs"])
assert torch.max(torch.abs(detect_scores - weights["detect_scores_ref"])) < 1e-5

pass_out = passport_layer_forward(
    weights["pass_x"], weights["pass_w"], weights["pass_b"], weights["passport_key"], weights["passport_weight"]
)
assert torch.max(torch.abs(pass_out - weights["passport_out_ref"])) < 1e-5
print("All Watermarking, Z-Score, Fast-DetectGPT, and Passport parity assertions passed!")

In [ ]:
# Benchmarking watermarking and obfuscation operators
t0 = time.perf_counter()
for _ in range(500):
    _ = watermark_logit_bias(weights["logits"], weights["green_mask"], 2.5)
    _ = watermark_z_score(weights["green_counts"], weights["seq_lens"], 0.5)
    _ = fast_detect_gpt_score(weights["log_probs"], weights["ref_log_probs"])
    _ = passport_layer_forward(weights["pass_x"], weights["pass_w"], weights["pass_b"], weights["passport_key"], weights["passport_weight"])
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Combined Watermarking, Detection, and Passport latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode & Security Auditing: Statistical Watermark vs Model Obfuscation
# 1. Watermark Detection: samples 0 and 2 have z > 4.0 (statistically verifiable watermarks)
for idx, z in enumerate(z_scores):
    status = "WATERMARKED (p < 1e-5)" if z.item() > 4.0 else "HUMAN / UNMARKED"
    print(f"Sample {idx}: Green Count = {weights['green_counts'][idx].item():.0f}/{weights['seq_lens'][idx].item():.0f}, z-score = {z.item():.2f} -> {status}")
assert z_scores[0].item() > 4.0 and z_scores[2].item() > 4.0, "Watermarked sequences must have z-score > 4.0!"

# 2. Passport Security: unauthorized key scrambles activations
fake_key = torch.randn(4) * 10.0
scrambled_out = passport_layer_forward(
    weights["pass_x"], weights["pass_w"], weights["pass_b"], fake_key, weights["passport_weight"]
)
diff_norm = torch.norm(pass_out - scrambled_out).item()
print(f"Norm discrepancy between authorized and unauthorized passport activations: {diff_norm:.4f}")
assert diff_norm > 0.2, "Unauthorized passport key must disrupt internal representations!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Green-List Biasing** inserts statistical watermarks into autoregressive text with analytical $z$-score hypothesis guarantees.
2. **Fast-DetectGPT** detects machine-generated text in 1 forward pass ($340\times$ speedup) by evaluating likelihood curvature.
3. **Passport Layers** protect proprietary neural weights from theft by conditioning internal gates on cryptographic secret keys.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [08_ai_watermarking_ai_detection_and_model_obfuscation.md](../../markdown/11_ai_safety_privacy_and_verification/08_ai_watermarking_ai_detection_and_model_obfuscation.md)
- **PyTorch 2.x Module:** [08_ai_watermarking_ai_detection_and_model_obfuscation.py](../../pytorch2/11_ai_safety_privacy_and_verification/08_ai_watermarking_ai_detection_and_model_obfuscation.py)
- **Functional JAX Module:** [08_ai_watermarking_ai_detection_and_model_obfuscation.py](../../jax/11_ai_safety_privacy_and_verification/08_ai_watermarking_ai_detection_and_model_obfuscation.py)
- **Native Rust Candle Track:** [08_ai_watermarking_ai_detection_and_model_obfuscation.rs](../../candle/11_ai_safety_privacy_and_verification/08_ai_watermarking_ai_detection_and_model_obfuscation.rs)
- **Hardware Kernel (CPU / WGSL):** [08_ai_watermarking_ai_detection_and_model_obfuscation.rs](../../kernels/11_ai_safety_privacy_and_verification/08_ai_watermarking_ai_detection_and_model_obfuscation.rs)
- **Golden SafeTensors:** [08_ai_watermarking_ai_detection_and_model_obfuscation.safetensors](../../weights/11_ai_safety_privacy_and_verification/08_ai_watermarking_ai_detection_and_model_obfuscation.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 07. Privacy DP SGD PATE Gradient Inversion DLG and MIA](07_privacy_dp_sgd_pate_gradient_inversion_dlg_and_mia.ipynb) | [Master Index](../../README.md#11_ai_safety_privacy_and_verification) | [Next: 09. OOD Detection Abstention Conformal and Algorithmic Fairness](09_ood_detection_abstention_conformal_and_algorithmic_fairness.ipynb) &rarr;